In [1]:
import pandas as pd
import unicodedata

In [2]:
rcp45 = pd.read_csv("rcp45_pr_BR_freq_intensity_2026-2050.csv", delimiter=",",encoding="utf-8")
rcp85 = pd.read_csv("rcp85_pr_BR_freq_intensity_2026-2050.csv", delimiter=",",encoding="utf-8")

In [5]:
rcp45.head(1)

,year,lat,lon,scenario,variable,wet_days,sdii,rx1day,rx5day,r20mm,r50mm,r95ptot_mm,r95ptot_frac,uf,municipio
0,2026,-33.6,-74.0,rcp45,pr,14,30.867908,53.628857,149.249023,14,1,296.689972,0.292613,RS,Barra do Quaraí


In [6]:
rcp85.head(1)

,year,lat,lon,scenario,variable,wet_days,sdii,rx1day,rx5day,r20mm,r50mm,r95ptot_mm,r95ptot_frac,uf,municipio
0,2026,-33.6,-74.0,rcp85,pr,8,25.296574,37.479015,66.34761,8,0,66.753174,0.102864,RS,Barra do Quaraí


In [7]:
for df in [rcp45, rcp85]:
    df["aux"] = df["municipio"] +"/" + df["uf"]

In [8]:
rcp85.head(1)

,year,lat,lon,scenario,variable,wet_days,sdii,rx1day,rx5day,r20mm,r50mm,r95ptot_mm,r95ptot_frac,uf,municipio,aux
0,2026,-33.6,-74.0,rcp85,pr,8,25.296574,37.479015,66.34761,8,0,66.753174,0.102864,RS,Barra do Quaraí,Barra do Quaraí/RS


In [9]:
rcp45 = rcp45[["year","aux","wet_days","sdii"]]
rcp85 = rcp85[["year","aux","wet_days","sdii"]]

In [10]:
rcp45.head()

,year,aux,wet_days,sdii
0,2026,Barra do Quaraí/RS,14,30.867908
1,2026,Barra do Quaraí/RS,15,29.498465
2,2026,Barra do Quaraí/RS,17,28.747995
3,2026,Barra do Quaraí/RS,17,29.930410
4,2026,Barra do Quaraí/RS,21,30.084707


In [11]:
rcp45 = rcp45.groupby(["aux","year"]).mean().reset_index()
rcp85 = rcp85.groupby(["aux","year"]).mean().reset_index()

In [12]:
rcp45.head()

,aux,year,wet_days,sdii
0,Abadia de Goiás/GO,2026,9.0,26.982172
1,Abadia de Goiás/GO,2027,16.0,27.451263
2,Abadia de Goiás/GO,2028,8.0,25.830688
3,Abadia de Goiás/GO,2029,9.0,27.766880
4,Abadia de Goiás/GO,2030,11.0,28.384619


In [17]:
rcp45["aux"] = rcp45["aux"].str.upper()
rcp85["aux"] = rcp85["aux"].str.upper()

In [18]:
base_fornecedores = pd.read_excel("Localizacao_fornecedores.xlsx", engine="openpyxl",header=None)    

In [19]:
def remover_acentos(texto):
    if isinstance(texto, str):
        return unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('ASCII')
    return texto  # retorna o valor original se não for string

In [20]:
base_fornecedores.head()

,0
0,MONTE BELO/MG
1,CONCEIÇÃO DO MATO DENTRO/MG
2,ARACAJU/SE
3,CAMBUCI/RJ
4,SÃO PAULO/SP


In [21]:
base_fornecedores.rename(columns={0: "aux"}, inplace=True)

In [22]:
base_fornecedores["aux"] = base_fornecedores["aux"].apply(remover_acentos)
base_fornecedores["aux"] = base_fornecedores["aux"].str.upper()

In [23]:
base_fornecedores.head()

,aux
0,MONTE BELO/MG
1,CONCEICAO DO MATO DENTRO/MG
2,ARACAJU/SE
3,CAMBUCI/RJ
4,SAO PAULO/SP


In [24]:
rcp45["aux"] = rcp45["aux"].apply(remover_acentos)
rcp85["aux"] = rcp85["aux"].apply(remover_acentos)

In [25]:
rcp45.head()

,aux,year,wet_days,sdii
0,ABADIA DE GOIAS/GO,2026,9.0,26.982172
1,ABADIA DE GOIAS/GO,2027,16.0,27.451263
2,ABADIA DE GOIAS/GO,2028,8.0,25.830688
3,ABADIA DE GOIAS/GO,2029,9.0,27.766880
4,ABADIA DE GOIAS/GO,2030,11.0,28.384619


In [26]:
locais = set(base_fornecedores["aux"])
locaisrcp45 = set(rcp45["aux"])
locaisrcp85 = set(rcp85["aux"])

In [27]:
len(locaisrcp85.intersection(locaisrcp45))

3969

In [28]:
len(locaisrcp45)

3969

Temos 14 locais não mapeados

In [29]:
len(locais) - len(locaisrcp45.intersection(locais))

14

In [30]:
nao_mapeados = locais - locaisrcp45.intersection(locais)

In [31]:
nao_mapeados

{'ALUMINIO/SP',
 'BARUERI/SP',
 'BENEVIDES/PA',
 'BENTO GONCALVES/RS',
 'CONTAGEM/MG',
 'EMBU DAS ARTES/SP',
 'HORTOLANDIA/SP',
 'ITAQUAQUECETUBA/SP',
 'OSASCO/SP',
 'PAULISTA/PE',
 'PORTO REAL/RJ',
 'RECIFE/PE',
 'SUZANO/SP',
 'VARZEA PAULISTA/SP'}

In [32]:
pd.Series(list(nao_mapeados)).to_csv("pontos_faltantes_fornecedores.csv", index=False, header=False, encoding="latin-1")